# Exploração do CNES - Leitos

Este notebook realiza a exploração inicial dos dados de leitos do Cadastro Nacional de Estabelecimentos de Saúde (CNES).

Nesta etapa serão analisados:

- arquivos mensais de leitos (`LT`)
- estrutura das colunas
- quantidade de registros
- dados ausentes
- estabelecimentos de saúde
- tipos de leito
- quantidade de leitos
- disponibilidade de leitos para o SUS

A análise será feita por ano, carregando automaticamente todos os arquivos mensais disponíveis.

## 1. Importação das bibliotecas

In [1]:
from pathlib import Path
import pandas as pd
from pysus.api.extensions import ExtensionFactory

## 2. Configuração da análise

O notebook procura automaticamente todos os arquivos `LT` disponíveis na pasta correspondente ao ano selecionado.

In [3]:
ANO = 2021
pasta_cnes = Path(f"../data/raw/cnes/{ANO}")
arquivos = sorted(pasta_cnes.glob("LTGO*.dbc"))

print(f"Ano analisado: {ANO}")
print(f"Arquivos encontrados: {len(arquivos)}")

for arquivo in arquivos:
    print(arquivo.name)

Ano analisado: 2021
Arquivos encontrados: 12
LTGO2101.dbc
LTGO2102.dbc
LTGO2103.dbc
LTGO2104.dbc
LTGO2105.dbc
LTGO2106.dbc
LTGO2107.dbc
LTGO2108.dbc
LTGO2109.dbc
LTGO2110.dbc
LTGO2111.dbc
LTGO2112.dbc


## 3. Validação dos arquivos mensais

Antes da leitura, verificamos quais meses estão disponíveis no conjunto de dados.

In [6]:
meses_encontrados = [ arquivo.stem[-2:] for arquivo in arquivos]

print("Meses encontrados:", meses_encontrados)

if len(arquivos) == 12:
    print("Todos os meses estão disponíveis.")
else:
    print("Atenção: o ano não possui 12 arquivos.")

Meses encontrados: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
Todos os meses estão disponíveis.


## 4. Leitura dos arquivos DBC

Os arquivos mensais serão carregados e reunidos em um único DataFrame para facilitar a exploração dos dados do ano.

In [7]:
async def carregar_dbc(caminho):
    # Carrega um arquivo DBC
    extensao = await ExtensionFactory.instantiate(caminho)
    df = await extensao.load()

    # Guarda o arquivo de origem
    df["ARQUIVO_ORIGEM"] = caminho.name

    return df

In [8]:
dataframes = []

for arquivo in arquivos:
    print(f"Carregando {arquivo.name}...")
    df_mes = await carregar_dbc(arquivo)
    dataframes.append(df_mes)

df_cnes = pd.concat( dataframes, ignore_index=True)

print("\nCarga concluída.")

Carregando LTGO2101.dbc...
Carregando LTGO2102.dbc...
Carregando LTGO2103.dbc...
Carregando LTGO2104.dbc...
Carregando LTGO2105.dbc...
Carregando LTGO2106.dbc...
Carregando LTGO2107.dbc...
Carregando LTGO2108.dbc...
Carregando LTGO2109.dbc...
Carregando LTGO2110.dbc...
Carregando LTGO2111.dbc...
Carregando LTGO2112.dbc...

Carga concluída.


## 5. Dimensão do dataset

Nesta etapa verificamos a quantidade de registros e colunas após reunir todos os meses de 2021.

In [9]:
linhas, colunas = df_cnes.shape

print(f"Registros: {linhas:,}")
print(f"Colunas: {colunas}")

Registros: 35,793
Colunas: 29


## 6. Visualização inicial

In [10]:
df_cnes.head()

,CNES,CODUFMUN,REGSAUDE,MICR_REG,DISTRSAN,DISTRADM,TPGESTAO,PF_PJ,CPF_CNPJ,NIV_DEP,...,TERCEIRO,TP_LEITO,CODLEITO,QT_EXIST,QT_CONTR,QT_SUS,QT_NSUS,COMPETEN,NAT_JUR,ARQUIVO_ORIGEM
0,9331603,520010,,,,,M,3,00000000000000,3,...,,2,33,9,0,9,0,202101,1244,LTGO2101.dbc
1,2335506,520013,15,,,,M,3,00269860000125,1,...,,6,34,4,0,3,1,202101,2062,LTGO2101.dbc
2,2335506,520013,15,,,,M,3,00269860000125,1,...,,1,03,2,0,1,1,202101,2062,LTGO2101.dbc
3,2335506,520013,15,,,,M,3,00269860000125,1,...,,5,45,3,0,3,0,202101,2062,LTGO2101.dbc
4,2335506,520013,15,,,,M,3,00269860000125,1,...,,4,43,3,0,3,0,202101,2062,LTGO2101.dbc


## 7. Colunas disponíveis

Antes de definir as variáveis utilizadas no projeto, verificamos os campos existentes nos arquivos de leitos do CNES.

In [11]:
for coluna in df_cnes.columns:
    print(coluna)

CNES
CODUFMUN
REGSAUDE
MICR_REG
DISTRSAN
DISTRADM
TPGESTAO
PF_PJ
CPF_CNPJ
NIV_DEP
CNPJ_MAN
ESFERA_A
ATIVIDAD
RETENCAO
NATUREZA
CLIENTEL
TP_UNID
TURNO_AT
NIV_HIER
TERCEIRO
TP_LEITO
CODLEITO
QT_EXIST
QT_CONTR
QT_SUS
QT_NSUS
COMPETEN
NAT_JUR
ARQUIVO_ORIGEM


In [16]:
palavras_chave = [
    "CNES",
    "LEIT",
    "SUS",
    "QT",
    "MUNIC",
    "UFMUN",
    "COMP",
    "TP",
    "ESP",
]

colunas_candidatas = [
    coluna
    for coluna in df_cnes.columns
    if any(
        palavra in coluna.upper()
        for palavra in palavras_chave
    )
]

colunas_candidatas

['CNES',
 'CODUFMUN',
 'TPGESTAO',
 'TP_UNID',
 'TP_LEITO',
 'CODLEITO',
 'QT_EXIST',
 'QT_CONTR',
 'QT_SUS',
 'QT_NSUS',
 'COMPETEN']

## 8. Estrutura dos dados

Nesta etapa verificamos os tipos das variáveis e uma amostra das colunas candidatas para a análise de capacidade hospitalar.

In [13]:
df_cnes[colunas_candidatas].head(10)

,CNES,TPGESTAO,TP_UNID,TP_LEITO,CODLEITO,QT_EXIST,QT_CONTR,QT_SUS,QT_NSUS,COMPETEN
0,9331603,M,15,2,33,9,0,9,0,202101
1,2335506,M,05,6,34,4,0,3,1,202101
2,2335506,M,05,1,03,2,0,1,1,202101
3,2335506,M,05,5,45,3,0,3,0,202101
4,2335506,M,05,4,43,3,0,3,0,202101
5,2335506,M,05,2,41,2,0,2,0,202101
6,2335506,M,05,2,33,6,0,4,2,202101
7,2335506,M,05,4,10,1,0,1,0,202101
8,2437570,M,05,2,33,16,0,16,0,202101
9,2437570,M,05,5,45,5,0,5,0,202101


In [14]:
df_cnes[colunas_candidatas].dtypes

CNES        string[python]
TPGESTAO    string[python]
TP_UNID     string[python]
TP_LEITO    string[python]
CODLEITO    string[python]
QT_EXIST    string[python]
QT_CONTR    string[python]
QT_SUS      string[python]
QT_NSUS     string[python]
COMPETEN    string[python]
dtype: object

## 9. Verificação de campos ausentes

Assim como no SIH/SUS, os arquivos podem representar valores ausentes como textos vazios.

Por isso, verificamos valores nulos e campos vazios.

In [15]:
resumo_cnes = pd.DataFrame( index=df_cnes.columns )
resumo_cnes["tipo"] = df_cnes.dtypes.astype(str)
resumo_cnes["nulos"] = ( df_cnes.isna().sum() )

resumo_cnes["vazios"] = df_cnes.apply(
    lambda coluna:
    coluna.astype(str)
    .str.strip()
    .eq("")
    .sum()
)

resumo_cnes["faltantes"] = ( resumo_cnes["nulos"] + resumo_cnes["vazios"])
resumo_cnes["faltantes_%"] = ( resumo_cnes["faltantes"] / len(df_cnes)* 100).round(2)
resumo_cnes["valores_unicos"] = ( df_cnes.nunique(dropna=True))

resumo_cnes.sort_values( "faltantes_%", ascending=False).head(30)

,tipo,nulos,vazios,faltantes,faltantes_%,valores_unicos
DISTRSAN,string,0,35793,35793,100.00,1
ESFERA_A,string,0,35793,35793,100.00,1
TERCEIRO,string,0,35793,35793,100.00,1
NATUREZA,string,0,35793,35793,100.00,1
RETENCAO,string,0,35793,35793,100.00,1
NIV_HIER,string,0,35793,35793,100.00,1
DISTRADM,string,0,35404,35404,98.91,5
MICR_REG,string,0,33939,33939,94.82,9
REGSAUDE,string,0,14016,14016,39.16,47
CLIENTEL,string,0,200,200,0.56,4


## 10. Seleção das variáveis de interesse

Para as análises de capacidade hospitalar serão utilizadas principalmente as variáveis relacionadas ao estabelecimento, município, tipo de leito, quantidade de leitos e competência.

In [17]:
colunas_interesse = [
    "CNES",
    "CODUFMUN",
    "TP_UNID",
    "TP_LEITO",
    "CODLEITO",
    "QT_EXIST",
    "QT_SUS",
    "QT_NSUS",
    "COMPETEN",
]

df_cnes_eda = df_cnes[colunas_interesse].copy()

df_cnes_eda.head()

,CNES,CODUFMUN,TP_UNID,TP_LEITO,CODLEITO,QT_EXIST,QT_SUS,QT_NSUS,COMPETEN
0,9331603,520010,15,2,33,9,9,0,202101
1,2335506,520013,05,6,34,4,3,1,202101
2,2335506,520013,05,1,03,2,1,1,202101
3,2335506,520013,05,5,45,3,3,0,202101
4,2335506,520013,05,4,43,3,3,0,202101


## 11. Conversão de tipos

Os arquivos DBC foram carregados com os campos em formato de texto. As quantidades de leitos serão convertidas para valores numéricos. Os códigos de estabelecimento, município e leito permanecerão como texto.

In [18]:
colunas_numericas = [
    "QT_EXIST",
    "QT_SUS",
    "QT_NSUS",
]

for coluna in colunas_numericas:
    df_cnes_eda[coluna] = pd.to_numeric(
        df_cnes_eda[coluna],
        errors="coerce"
    )

In [19]:
df_cnes_eda["ANO"] = (
    df_cnes_eda["COMPETEN"]
    .str[:4]
    .astype(int)
)

df_cnes_eda["MES"] = (
    df_cnes_eda["COMPETEN"]
    .str[4:6]
    .astype(int)
)

df_cnes_eda.head()

,CNES,CODUFMUN,TP_UNID,TP_LEITO,CODLEITO,QT_EXIST,QT_SUS,QT_NSUS,COMPETEN,ANO,MES
0,9331603,520010,15,2,33,9,9,0,202101,2021,1
1,2335506,520013,05,6,34,4,3,1,202101,2021,1
2,2335506,520013,05,1,03,2,1,1,202101,2021,1
3,2335506,520013,05,5,45,3,3,0,202101,2021,1
4,2335506,520013,05,4,43,3,3,0,202101,2021,1


## 12. Verificação de registros repetidos

Antes de somar a quantidade de leitos, verificamos se existem registros repetidos para o mesmo estabelecimento, tipo de leito, código de leito e competência.

In [21]:
chave_leito = [
    "CNES",
    "TP_LEITO",
    "CODLEITO",
    "COMPETEN",
]

duplicados_leito = df_cnes_eda.duplicated( subset=chave_leito, keep=False )
print( "Registros envolvidos em chaves repetidas:", duplicados_leito.sum())

Registros envolvidos em chaves repetidas: 0


In [22]:
df_cnes_eda[duplicados_leito].sort_values( chave_leito ).head(30)

,CNES,CODUFMUN,TP_UNID,TP_LEITO,CODLEITO,QT_EXIST,QT_SUS,QT_NSUS,COMPETEN,ANO,MES


## 13. Consistência das quantidades de leitos

Nesta etapa verificamos se a quantidade total de leitos existentes corresponde à soma dos leitos SUS e não SUS.

In [ ]:
df_cnes_eda["DIFERENCA_LEITOS"] = (
    df_cnes_eda["QT_EXIST"] - ( df_cnes_eda["QT_SUS"] + df_cnes_eda["QT_NSUS"] )
)

df_cnes_eda["DIFERENCA_LEITOS"].value_counts().head(20)

DIFERENCA_LEITOS
0    35793
Name: count, dtype: Int64

## 14. Resumo mensal da capacidade hospitalar

Após validar os registros, será observado o comportamento da quantidade de estabelecimentos e leitos ao longo das competências de 2021.

In [24]:
resumo_mensal_cnes = (
    df_cnes_eda
    .groupby(["ANO", "MES"])
    .agg(
        estabelecimentos=("CNES", "nunique"),
        municipios=("CODUFMUN", "nunique"),
        leitos_existentes=("QT_EXIST", "sum"),
        leitos_sus=("QT_SUS", "sum"),
        leitos_nao_sus=("QT_NSUS", "sum"),
    )
    .reset_index()
)

resumo_mensal_cnes

,ANO,MES,estabelecimentos,municipios,leitos_existentes,leitos_sus,leitos_nao_sus
0,2021,1,489,187,21187,12776,8411
1,2021,2,491,187,21592,12854,8738
2,2021,3,501,189,22066,13506,8560
3,2021,4,500,189,22098,13301,8797
4,2021,5,503,189,22169,13341,8828
5,2021,6,507,190,22323,13459,8864
6,2021,7,509,190,22525,13547,8978
7,2021,8,514,191,22722,13746,8976
8,2021,9,512,192,22758,13853,8905
9,2021,10,514,193,22833,13984,8849


## 15. Participação dos leitos SUS

Nesta etapa verificamos a participação dos leitos destinados ao SUS em relação ao total de leitos existentes em cada competência.

In [25]:
participacao_sus = resumo_mensal_cnes.copy()

participacao_sus["percentual_sus"] = ( participacao_sus["leitos_sus"] / participacao_sus["leitos_existentes"]* 100).round(2)

participacao_sus[
    [
        "ANO",
        "MES",
        "leitos_existentes",
        "leitos_sus",
        "percentual_sus",
    ]
]

,ANO,MES,leitos_existentes,leitos_sus,percentual_sus
0,2021,1,21187,12776,60.3
1,2021,2,21592,12854,59.53
2,2021,3,22066,13506,61.21
3,2021,4,22098,13301,60.19
4,2021,5,22169,13341,60.18
5,2021,6,22323,13459,60.29
6,2021,7,22525,13547,60.14
7,2021,8,22722,13746,60.5
8,2021,9,22758,13853,60.87
9,2021,10,22833,13984,61.24


## 16. Capacidade por tipo de leito

A quantidade de leitos será analisada de acordo com o tipo de leito cadastrado no CNES.

In [28]:
leitos_por_tipo_mes = (
    df_cnes_eda
    .groupby(
        ["ANO", "MES", "TP_LEITO"]
    )
    .agg(
        leitos_existentes=("QT_EXIST", "sum"),
        leitos_sus=("QT_SUS", "sum"),
        leitos_nao_sus=("QT_NSUS", "sum"),
    )
    .reset_index()
)

leitos_por_tipo_mes.head(20)

,ANO,MES,TP_LEITO,leitos_existentes,leitos_sus,leitos_nao_sus
0,2021,1,1,4909,2748,2161
1,2021,1,2,6547,4654,1893
2,2021,1,3,2897,1323,1574
3,2021,1,4,2074,1375,699
4,2021,1,5,1725,1198,527
5,2021,1,6,2601,1252,1349
6,2021,1,7,434,226,208
7,2021,2,1,4888,2734,2154
8,2021,2,2,6676,4769,1907
9,2021,2,3,3206,1307,1899


## 17. Estabelecimentos com maior capacidade SUS

Nesta etapa identificamos os estabelecimentos com maior quantidade de leitos SUS em uma competência específica.

In [29]:
ultima_competencia = df_cnes_eda["COMPETEN"].max()

capacidade_estabelecimentos = (
    df_cnes_eda[
        df_cnes_eda["COMPETEN"] == ultima_competencia
    ]
    .groupby(["CNES", "CODUFMUN"])
    .agg(
        leitos_existentes=("QT_EXIST", "sum"),
        leitos_sus=("QT_SUS", "sum"),
        leitos_nao_sus=("QT_NSUS", "sum"),
    )
    .reset_index()
    .sort_values(
        "leitos_sus",
        ascending=False
    )
)

print("Competência analisada:", ultima_competencia)
capacidade_estabelecimentos.head(20)

Competência analisada: 202112


,CNES,CODUFMUN,leitos_existentes,leitos_sus,leitos_nao_sus
466,7743068,520870,485,469,16
511,9680977,520140,390,359,31
48,2338262,520870,365,357,8
308,2535939,522140,373,300,73
54,2338424,520870,421,265,156
49,2338351,520870,334,256,78
1,0086126,520870,293,243,50
58,2338734,520870,237,237,0
239,2517957,520870,307,193,114
351,2673932,520870,196,176,20


## 18. Capacidade SUS por município

A capacidade hospitalar será agregada por município utilizando a competência mais recente disponível no ano analisado.

In [30]:
capacidade_municipios = (
    df_cnes_eda[
        df_cnes_eda["COMPETEN"] == ultima_competencia
    ]
    .groupby("CODUFMUN")
    .agg(
        estabelecimentos=("CNES", "nunique"),
        leitos_existentes=("QT_EXIST", "sum"),
        leitos_sus=("QT_SUS", "sum"),
        leitos_nao_sus=("QT_NSUS", "sum"),
    )
    .reset_index()
    .sort_values(
        "leitos_sus",
        ascending=False
    )
)

capacidade_municipios.head(20)

,CODUFMUN,estabelecimentos,leitos_existentes,leitos_sus,leitos_nao_sus
77,520870,132,9306,4161,5145
13,520140,19,1700,1337,363
11,520110,19,1364,748,616
183,522140,5,555,444,111
157,521880,12,648,400,248
187,522160,5,402,252,150
43,520510,6,397,218,179
76,520860,6,283,195,88
116,521310,7,281,191,90
108,521250,9,272,180,92


## 19. Conclusão

A exploração dos dados de leitos do CNES de 2021 permitiu identificar a estrutura da capacidade hospitalar disponível no período.

Os principais resultados observados foram:

- foram carregados os 12 arquivos mensais de leitos de 2021;
- o conjunto possui 35.793 registros;
- foram identificados 527 estabelecimentos CNES ao longo do ano;
- os campos necessários para analisar capacidade hospitalar não apresentam valores ausentes;
- não foram identificadas duplicidades na chave formada por estabelecimento, tipo de leito, código de leito e competência;
- em todos os registros, a quantidade de leitos existentes corresponde à soma dos leitos SUS e não SUS;
- a capacidade hospitalar variou ao longo das competências de 2021;
- entre janeiro e dezembro houve crescimento no número de estabelecimentos e na quantidade de leitos cadastrados;
- os dados permitem analisar capacidade por estabelecimento, município e tipo de leito.

Os resultados serão utilizados posteriormente na integração com os dados do SIH/SUS para comparar capacidade hospitalar e demanda por internações.